# FFTop GPU Kernel Sweep (Colab)

Runtime → **Change runtime type → GPU** (T4 is enough).

File → Open notebook → GitHub → `nguyen-trinhtk/FFTop` → branch **`colab-gpu`** → `gpu-notebooks/stockham_sweep_colab.ipynb`.
The clone cell pulls `https://github.com/nguyen-trinhtk/FFTop.git` (`colab-gpu`) into `/content/FFTop`.

Builds FFTop with CUDA and sweeps three kernel strategies:

- **cooley-tukey naive** — in-place Cooley–Tukey in global memory
- **stockham naive** — self-sorting Stockham, ping-pong global memory
- **stockham shared** — Stockham in shared memory (four-step when \(N\) exceeds the tile)

Metrics (device kernels only, no H2D/D2H):

- runtime (ms)
- effective bandwidth (GB/s) = estimated global traffic / time
- estimated global memory traffic (bytes)
- throughput (Gsamples/s) = \(N / t\)

In [ ]:
!nvidia-smi

In [ ]:
!apt-get -qq update
!apt-get -qq install -y cmake ninja-build

In [ ]:
import os
import subprocess
from pathlib import Path

ROOT = Path("/content/FFTop")
REPO_URL = "https://github.com/nguyen-trinhtk/FFTop.git"
BRANCH = "colab-gpu"

env = os.environ.copy()
env["GIT_TERMINAL_PROMPT"] = "0"

if ROOT.exists() and not (ROOT / ".git").exists():
    subprocess.check_call(["rm", "-rf", str(ROOT)])

if ROOT.exists():
    subprocess.check_call(["git", "-C", str(ROOT), "fetch", "origin"], env=env)
    subprocess.check_call(["git", "-C", str(ROOT), "checkout", BRANCH], env=env)
    subprocess.check_call(["git", "-C", str(ROOT), "pull", "--ff-only", "origin", BRANCH], env=env)
else:
    subprocess.check_call(
        ["git", "clone", "--branch", BRANCH, REPO_URL, str(ROOT)], env=env
    )

os.chdir(ROOT)
print("repo:", ROOT.resolve())
subprocess.check_call(["git", "rev-parse", "--short", "HEAD"])
subprocess.check_call(["git", "status", "-sb"])

In [ ]:
import subprocess
from pathlib import Path

cap = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
    text=True,
).strip().splitlines()[0]
arch = cap.replace(".", "")
print("CUDA arch:", arch)

subprocess.check_call([
    "cmake", "-S", ".", "-B", "build", "-G", "Ninja",
    "-DCMAKE_BUILD_TYPE=Release",
    "-DFFTOP_ENABLE_CUDA=ON",
    f"-DCMAKE_CUDA_ARCHITECTURES={arch}",
])
subprocess.check_call(["cmake", "--build", "build", "--target", "fftop_bench", "fftop_tests", "-j"])

bench = Path("build/bench/fftop_bench")
assert bench.is_file(), f"missing {bench.resolve()} — CUDA build failed"
print("bench:", bench.resolve())

In [ ]:
import subprocess
subprocess.check_call(["python3", "-m", "pip", "-q", "install", "-r", "bench/requirements.txt"])

In [ ]:
import subprocess
from pathlib import Path

import yaml

# Shrink the sweep on a slow GPU. Default yaml is min_k=10, max_k=24.
MAX_K = 20

cfg_path = Path("bench/config/gpu_stockham.yaml")
assert cfg_path.is_file(), f"not in repo root? cwd={Path.cwd()}"
cfg = yaml.safe_load(cfg_path.read_text())
cfg.setdefault("defaults", {})["max_k"] = MAX_K
run_cfg = Path("/tmp/gpu_stockham_colab.yaml")
run_cfg.write_text(yaml.safe_dump(cfg, sort_keys=False))

bench = Path("build/bench/fftop_bench")
assert bench.is_file(), f"missing {bench} — rerun the cmake cell"

subprocess.check_call(["python3", "bench/bench.py", "--check", "--config", str(run_cfg)])
subprocess.check_call(["ctest", "--test-dir", "build", "--output-on-failure", "-R", "GPUBackend"])
subprocess.check_call([
    "python3", "bench/bench.py",
    "--bench", str(bench),
    "--config", str(run_cfg),
])

csvs = sorted(Path("log").glob("*/bench.csv"))
assert csvs, "sweep wrote no bench.csv — scroll up for the bench.py / fftop_bench error"
print("wrote", csvs[-1])

In [ ]:
import pandas as pd
from pathlib import Path

csvs = sorted(Path("log").glob("*/bench.csv"))
if not csvs:
    logs = sorted(Path("log").glob("*"))
    print("cwd:", Path.cwd())
    print("log dirs:", logs)
    if logs:
        print("latest dir:", logs[-1], "->", list(logs[-1].iterdir()))
    raise FileNotFoundError(
        "no log/*/bench.csv. Re-run the sweep cell and read its traceback "
        "(build/bench/fftop_bench missing or fftop_bench crashed)."
    )

csv_path = csvs[-1]
log_dir = csv_path.parent
df = pd.read_csv(csv_path, comment="#")
df["k"] = df["size"].map(lambda n: int(n).bit_length() - 1)
print("Using:", csv_path)
df.head()

In [ ]:
import matplotlib.pyplot as plt

name_map = {
    "cooley-tukey": "cooley-tukey naive",
    "stockham-global": "stockham naive",
    "stockham-shared": "stockham shared",
}
df["variant"] = df["traversal"].map(name_map).fillna(df["traversal"])

metrics = [
    ("ms", "Runtime (ms)", True),
    ("effective_bandwidth_gbps", "Effective Bandwidth (GB/s)", False),
    ("throughput_gsamples_s", "Throughput (Gsamples/s)", False),
    ("global_mem_bytes", "Estimated Global Memory Traffic (bytes)", False),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
axes = axes.ravel()
for ax, (col, title, invert) in zip(axes, metrics):
    for name, g in df.groupby("variant"):
        g = g.sort_values("size")
        ax.plot(g["size"], g[col], marker="o", label=name)
    ax.set_xscale("log", base=2)
    if col == "global_mem_bytes":
        ax.set_yscale("log")
    ax.set_title(title)
    ax.set_xlabel("N")
    ax.grid(True, which="both", linestyle=":", linewidth=0.6)
    if invert:
        ax.set_ylabel("lower is better")
    else:
        ax.set_ylabel("higher is better")
axes[0].legend()
plt.show()

In [ ]:
pivot_cols = ["ms", "effective_bandwidth_gbps", "throughput_gsamples_s", "global_mem_bytes"]
show_n = [2**k for k in (12, 16, 20) if 2**k in set(df["size"])]
view = df[df["size"].isin(show_n)][["variant", "size", *pivot_cols]].sort_values(["size", "variant"])
view

In [ ]:
from IPython.display import Image, display

metrics_png = log_dir / "gpu-kernel-strategies_metrics.png"
if metrics_png.is_file():
    display(Image(filename=str(metrics_png)))
else:
    print("no combined plot at", metrics_png)